# Silver DOT Overnight Travelers

This notebook turns the DOT **Regional Distribution of Overnight Travelers** Bronze tables (DS_05, one per year) into travelers per **reporting unit and year**, using the reviewed crosswalk from `01_silver_geography`.

| Silver table | One row = | Used by |
| --- | --- | --- |
| `dot_travelers_row` | one DOT region or province-level row (indent level 0 or 1) per year, with numbers cleaned and its crosswalk role | traceability, reconciliation, `dot_excluded_rows` |
| `dot_travelers_unit_year` | one reporting unit per year (100 units × every DOT year) | Gold `fact_province_year` (tourism columns) |
| `dot_excluded_rows` | one DOT row that is not loaded into any unit (SBMA / Subic Bay Freeport Zone, Cotabato City, Boracay double counts) | exceptions list in the report |

**Rules applied** (decision log):

* **Row roles** from `map_dot_area_to_psgc` (DL-010 to DL-018): `province_unit` and `add_to_parent` rows are summed into their unit; `exclude` rows are kept only in `dot_excluded_rows`; region and `GRAND TOTAL` rows are used only for reconciliation. Level-2 sub-rows are never loaded (they are already inside their parent).
* **The `-` marker** (DL-019): a `-` cell in a row whose total is printed is 0; a row whose total is `-` is missing. Each unit-year gets `traveler_value_status`:

| `traveler_value_status` | Meaning | Values |
| --- | --- | --- |
| `reported` | every contributing row printed, no `-` cells | sums |
| `dash_as_zero` | every contributing row printed, some `-` cells counted as 0 | sums |
| `reported_partial` | some contributing rows have a `-` total (NCR in every year, Sorsogon 2019) | sums of the printed rows |
| `not_reported` | every contributing row has a `-` total, or DOT has no row for the unit (BARMM provinces, Sulu) | NULL |

* **Comparability with 2019** (DL-020): `comparable_to_2019 = false` in a year when any crosswalk row that belongs to the unit in that year has `breaks_comparability = true` (DOT counts it differently from 2019). Currently only Aklan, 2021–2024 (DL-013).
* **Region II** prints foreign and overseas Filipino travelers together; those units get `foreign_includes_overseas_filipinos = true` so the split is not compared with other regions.

Every data-quality check runs **before** writing. If a `BLOCK` check fails, results go to `dq_results`, no Silver table is written, and the notebook raises an error.

## Configuration

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window
import uuid, re
from datetime import datetime

# ── Unity Catalog targets ────────────────────────────────────────
CATALOG = "tuklas_dev"
BRONZE_SCHEMA = "02_bronze"
SILVER_SCHEMA = "03_silver"
DQ_SCHEMA = "06_data_quality"
NOTEBOOK_NAME = "02_silver_dot"

run_id = f"silver_dot_{datetime.now().strftime('%Y%m%d_%H%M%S')}_{uuid.uuid4().hex[:8]}"

# ── Inputs ───────────────────────────────────────────────────────
DOT_TABLE_RE = re.compile(r"^dot_ds_05_(\d{4})$")   # 03_ingest_dot, one table per year
TBL_UNIT = "geo_reporting_unit"                      # 01_silver_geography
TBL_PSGC = "geo_psgc"
TBL_DOT_MAP = "map_dot_area_to_psgc"
MEASURES = ["foreign_travelers", "overseas_filipinos", "domestic_travelers", "total_travelers"]

# ── Rules and expected results (change only through the decision log) ──
REGION_II_LABEL_RE = r"^Region II \("            # DOT's Region II rows: foreign and overseas Filipinos printed together
YOY_RATIO_REVIEW = 5.0                           # DQ-DOT-11: year-over-year change above 5x (or below 1/5) is listed for review
PANDEMIC_SWING_YEARS = [2020, 2021, 2022]        # changes INTO these years are expected to be large (pandemic); not flagged
EXPECTED_UNITS_WITHOUT_DOT = {"0906600000", "1900700000", "1903600000", "1907000000", "1908700000", "1908800000"}
EXPECTED_NOT_COMPARABLE = {("0600400000", y) for y in (2021, 2022, 2023, 2024)}   # Aklan (DL-013, DL-020)

# ── Output tables ────────────────────────────────────────────────
TBL_ROWS = "dot_travelers_row"
TBL_UNIT_YEAR = "dot_travelers_unit_year"
TBL_EXCLUDED = "dot_excluded_rows"

def bronze_table(name): return f"`{CATALOG}`.`{BRONZE_SCHEMA}`.`{name}`"
def silver_table(name): return f"`{CATALOG}`.`{SILVER_SCHEMA}`.`{name}`"
def dq_table(name): return f"`{CATALOG}`.`{DQ_SCHEMA}`.`{name}`"

print(f"Run ID: {run_id}")

## Helpers

Same pattern as `01_silver_geography`: `dq(...)` records each check in memory (written to `dq_results` at the end), and `write_silver(...)` overwrites a whole Silver table, so a rerun gives the same result.

In [0]:
dq_results = []

def dq(check_id, table_name, severity, failed_rows, expected, observed, sample_keys=None, notes=""):
    """Record one data-quality check result. failed_rows = 0 means PASS."""
    status = "PASS" if failed_rows == 0 else "FAIL"
    keys = " | ".join(str(k) for k in list(sample_keys)[:20]) if sample_keys else None
    dq_results.append((run_id, NOTEBOOK_NAME, check_id, table_name, severity, status, int(failed_rows),
                       str(expected), str(observed), keys, notes))
    marker = "✓" if status == "PASS" else ("✗" if severity == "BLOCK" else "!")
    print(f"{marker} {check_id:<11} {severity:<6} {status:<4} | {table_name} | expected {expected} | observed {observed}"
          + (f" | e.g. {keys}" if keys else ""))

def sample(df, *cols, n=20):
    return [" / ".join("" if r[c] is None else str(r[c]) for c in cols) for r in df.select(*cols).limit(n).collect()]

def write_silver(df, name):
    (df.write.format("delta").mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(silver_table(name)))
    return spark.table(silver_table(name)).count()

spark.sql(f"""
INSERT INTO {silver_table('silver_run_log')}
(run_id, notebook_name, started_at, completed_at, tables_written, rows_written,
 dq_checks_run, dq_block_failures, dq_review_flags, status, notes)
VALUES (:run_id, :nb, current_timestamp(), NULL, 0, 0, 0, 0, 0, 'STARTED', 'Notebook execution started')
""", args={"run_id": run_id, "nb": NOTEBOOK_NAME})
print(f"✓ STARTED record inserted for {run_id}")

## Step 1 — `dot_travelers_row`: clean each DOT row and attach its crosswalk role

From every `dot_ds_05_<year>` table, rows at indent level 0 (regions, `GRAND TOTAL`) and 1 (provinces, HUCs, cities, zones) are kept. Each value printed as `1,234` becomes a whole number; `-` is kept as a flag (`*_is_dash`), so the `-` rule can be applied per row and per unit. A value that is neither a number nor `-` stays NULL and fails DQ-COM-06.

The crosswalk is matched exactly as in `01_silver_geography` (indent level, parent label, area label, year within the validity period). For `exclude` rows that have no `unit_psgc_code` (e.g. Boracay 2021–2024), the unit they belong to is taken from the PSGC unit of the printed area (Malay → Aklan), so comparability can still be traced to the right unit.

In [0]:
dot_tables = sorted(r["tableName"] for r in spark.sql(f"SHOW TABLES IN `{CATALOG}`.`{BRONZE_SCHEMA}`").collect()
                    if DOT_TABLE_RE.match(r["tableName"]))
assert dot_tables, "No dot_ds_05_<year> tables found in Bronze. Run 03_ingest_dot first."
print(f"DOT Bronze tables: {dot_tables}")

bronze = None
for t in dot_tables:
    df = (spark.table(bronze_table(t))
          .filter(F.col("indent_level").isin("0", "1"))
          .select(F.col("source_year").cast("int").alias("year"),
                  F.col("page_number").cast("int").alias("page_number"),
                  F.col("pdf_line_number").cast("int").alias("pdf_line_number"),
                  F.col("indent_level").cast("int").alias("indent_level"),
                  F.trim("parent_label").alias("parent_label"),
                  F.trim("area_label").alias("area_label"),
                  *[F.trim(F.col(m)).alias(f"{m}_text") for m in MEASURES],
                  "_source_dataset_id", "_source_file", "_source_sha256",
                  F.col("_run_id").alias("_bronze_run_id"),
                  F.lit(t).alias("bronze_table")))
    bronze = df if bronze is None else bronze.unionByName(df)

bronze = bronze.withColumn("parent_label", F.when(F.col("indent_level") == 0, F.lit(None)).otherwise(F.col("parent_label")))
for m in MEASURES:
    txt = F.col(f"{m}_text")
    bronze = (bronze
        .withColumn(f"{m}_is_dash", txt == "-")
        .withColumn(m, F.when(txt == "-", F.lit(0).cast("bigint"))
                        .otherwise(F.expr(f"try_cast(regexp_replace({m}_text, ',', '') AS BIGINT)"))))

dot_map = spark.table(silver_table(TBL_DOT_MAP))
area_unit = spark.table(silver_table(TBL_PSGC)).select(F.col("psgc_code").alias("area_psgc_code"),
                                                     F.col("unit_psgc_code").alias("_area_unit"))
s = (dot_map.join(area_unit, "area_psgc_code", "left")
     .select("dot_indent_level", "dot_parent_label", "dot_area_label", "valid_from_year", "valid_to_year",
             "row_role", "unit_psgc_code", "area_psgc_code", "exclude_reason", "breaks_comparability",
             F.coalesce("unit_psgc_code", "_area_unit").alias("belongs_to_unit"))
     .alias("s"))
b = bronze.alias("b")
rows = (b.join(s,
        (F.col("b.indent_level") == F.col("s.dot_indent_level"))
        & F.col("b.parent_label").eqNullSafe(F.col("s.dot_parent_label"))
        & (F.col("b.area_label") == F.col("s.dot_area_label"))
        & (F.col("b.year") >= F.col("s.valid_from_year"))
        & (F.col("b.year") <= F.coalesce(F.col("s.valid_to_year"), F.lit(9999))),
        "left")
    .select("b.year", "b.page_number", "b.pdf_line_number", "b.indent_level", "b.parent_label", "b.area_label",
            *[F.col(f"b.{m}") for m in MEASURES], *[F.col(f"b.{m}_is_dash") for m in MEASURES],
            *[F.col(f"b.{m}_text") for m in MEASURES],
            "s.row_role", "s.unit_psgc_code", "s.area_psgc_code", "s.belongs_to_unit", "s.exclude_reason",
            "s.breaks_comparability", F.col("s.valid_from_year").alias("map_valid_from_year"),
            "b._source_dataset_id", "b._source_file", "b._source_sha256", "b._bronze_run_id", "b.bronze_table")
    .withColumn("row_total_is_dash", F.col("total_travelers_is_dash"))
    .withColumn("dash_cells", sum(F.col(f"{m}_is_dash").cast("int") for m in MEASURES))
    .withColumn("foreign_includes_overseas_filipinos",
                (F.col("indent_level") == 1) & F.col("parent_label").rlike(REGION_II_LABEL_RE))
    .withColumn("_silver_run_id", F.lit(run_id))
    .withColumn("_processed_at", F.current_timestamp()))

dot_years = sorted(r["year"] for r in rows.select("year").distinct().collect())
print(f"DOT years: {dot_years} | level 0/1 rows: {rows.count()}")
display(rows.groupBy("year", "row_role").count().orderBy("year", "row_role"))

## Step 2 — `dot_travelers_unit_year`: one row per unit and year

The grid is every reporting unit (100) × every DOT year, so units DOT does not report still get a row (`not_reported`, NULL values).

For each unit-year, the `province_unit` and `add_to_parent` rows are combined: values are summed over the rows whose total is printed (`-` cells count as 0), and the status follows DL-019. A row whose total is `-` contributes nothing.

In [0]:
loaded = rows.filter(F.col("row_role").isin("province_unit", "add_to_parent"))

agg = (loaded.groupBy(F.col("unit_psgc_code"), "year")
    .agg(F.count("*").alias("contributing_rows"),
         F.sum(F.col("row_total_is_dash").cast("int")).alias("dash_total_rows"),
         F.sum(F.when(~F.col("row_total_is_dash"), F.col("dash_cells")).otherwise(0)).alias("dash_cells_in_printed_rows"),
         *[F.sum(F.when(~F.col("row_total_is_dash"), F.col(m))).alias(m) for m in MEASURES],
         F.max(F.col("foreign_includes_overseas_filipinos").cast("int")).cast("boolean").alias("foreign_includes_overseas_filipinos"),
         F.concat_ws("; ", F.array_sort(F.collect_set("area_label"))).alias("dot_labels"),
         F.min("_source_dataset_id").alias("_source_dataset_id"),
         F.concat_ws("; ", F.array_sort(F.collect_set("_source_file"))).alias("_source_file"),
         F.concat_ws("; ", F.array_sort(F.collect_set("_source_sha256"))).alias("_source_sha256"),
         F.concat_ws("; ", F.array_sort(F.collect_set("_bronze_run_id"))).alias("_bronze_run_id")))

# Comparability: any crosswalk row belonging to the unit in that year (loaded or excluded) that breaks comparability
breaks = (rows.filter(F.col("breaks_comparability") & F.col("belongs_to_unit").isNotNull())
          .groupBy(F.col("belongs_to_unit").alias("unit_psgc_code"), "year")
          .agg(F.concat_ws("; ", F.array_sort(F.collect_set("area_label"))).alias("_break_labels")))

units = spark.table(silver_table(TBL_UNIT)).select("unit_psgc_code", "unit_name", "unit_type", "region_code", "region_name")
grid = units.crossJoin(spark.createDataFrame([(y,) for y in dot_years], "year INT"))

status = (F.when(F.col("contributing_rows").isNull(), F.lit("not_reported"))
           .when(F.col("dash_total_rows") == F.col("contributing_rows"), F.lit("not_reported"))
           .when(F.col("dash_total_rows") > 0, F.lit("reported_partial"))
           .when(F.col("dash_cells_in_printed_rows") > 0, F.lit("dash_as_zero"))
           .otherwise(F.lit("reported")))

unit_year = (grid
    .join(agg, ["unit_psgc_code", "year"], "left")
    .join(breaks, ["unit_psgc_code", "year"], "left")
    .withColumn("traveler_value_status", status)
    .withColumn("comparable_to_2019", F.col("_break_labels").isNull())
    .withColumn("comparability_note", F.when(F.col("_break_labels").isNotNull(),
                F.concat(F.lit("DOT counts this unit differently from 2019 (rows: "), F.col("_break_labels"), F.lit(")"))))
    .withColumn("contributing_rows", F.coalesce("contributing_rows", F.lit(0)))
    .withColumn("dash_total_rows", F.coalesce("dash_total_rows", F.lit(0)))
    .withColumn("foreign_includes_overseas_filipinos", F.coalesce("foreign_includes_overseas_filipinos", F.lit(False)))
    .withColumn("_silver_run_id", F.lit(run_id))
    .withColumn("_processed_at", F.current_timestamp()))
for m in MEASURES:
    unit_year = unit_year.withColumn(m, F.when(F.col("traveler_value_status") == "not_reported", F.lit(None).cast("bigint"))
                                          .otherwise(F.col(m)))
unit_year = unit_year.select(
    "unit_psgc_code", "unit_name", "unit_type", "region_code", "region_name", "year", *MEASURES,
    "traveler_value_status", "comparable_to_2019", "comparability_note", "foreign_includes_overseas_filipinos",
    "contributing_rows", "dash_total_rows", "dash_cells_in_printed_rows", "dot_labels",
    "_source_dataset_id", "_source_file", "_source_sha256", "_bronze_run_id", "_silver_run_id", "_processed_at")

display(unit_year.groupBy("traveler_value_status").count().orderBy("traveler_value_status"))

## Step 3 — `dot_excluded_rows`

Every `exclude` row, with its reason, so the report can list what is not assigned to any unit (DL-015, DL-016) and what was left out to avoid double counting (DL-013).

In [0]:
excluded = (rows.filter(F.col("row_role") == "exclude")
    .select("year", "area_label", "parent_label", "exclude_reason", "belongs_to_unit", "area_psgc_code",
            *MEASURES, "row_total_is_dash", "page_number", "pdf_line_number",
            "_source_dataset_id", "_source_file", "_source_sha256", "_bronze_run_id", "_silver_run_id", "_processed_at"))
display(excluded.groupBy("exclude_reason", "area_label").agg(F.sum("total_travelers").alias("travelers")).orderBy("exclude_reason", "area_label"))

## Step 4 — Data-quality checks

IDs follow `docs/data-quality-rules.md` (Issue #5).

| ID | Check | Severity |
| --- | --- | --- |
| DQ-COM-03 | One row per unit and year; row count = units × years | BLOCK |
| DQ-COM-04 | Lineage filled on every unit-year with values | BLOCK |
| DQ-COM-06 | Every printed DOT value is a number or `-` | BLOCK |
| DQ-COM-07 | Values are NULL exactly when the status is `not_reported`; status uses allowed values | BLOCK |
| DQ-DOT-01 | Domestic + foreign + overseas Filipino = total in every printed row | BLOCK |
| DQ-DOT-03 | Every level-0/1 row has a crosswalk role | BLOCK |
| DQ-DOT-04 | Only level-0/1 rows are used (no sub-rows) | BLOCK |
| DQ-DOT-07 | Aklan counts Boracay once: 2019 = Aklan + Boracay rows; 2023 = Aklan row only | BLOCK |
| DQ-DOT-09 | BARMM provinces and Sulu are `not_reported`, never 0 | BLOCK |
| DQ-DOT-10 | Region II units carry the foreign / overseas Filipino flag | LOG |
| DQ-DOT-11 | Year-over-year change above 5× (or below 1/5), outside the pandemic swing (changes into 2020–2022 are expected) | REVIEW |
| DQ-DOT-15 | Per year: units + non-double-count exclusions = DOT `GRAND TOTAL` | BLOCK |
| DQ-DOT-16 | Status counts: 28 `not_reported` from `-` rows, 7 `reported_partial` (DL-019) | REVIEW |
| DQ-DOT-17 | `comparable_to_2019 = false` only for the expected unit-years (Aklan 2021–2024) | REVIEW |

In [0]:
n_units = units.count()
dup = unit_year.groupBy("unit_psgc_code", "year").count().filter("count > 1")
n_uy = unit_year.count()
dq("DQ-COM-03", TBL_UNIT_YEAR, "BLOCK", dup.count() + (0 if n_uy == n_units * len(dot_years) else 1),
   f"{n_units} units x {len(dot_years)} years = {n_units * len(dot_years)} rows, no duplicates",
   f"{n_uy} rows, {dup.count()} duplicate keys", sample(dup, "unit_psgc_code", "year"))

no_lineage = unit_year.filter((F.col("traveler_value_status") != "not_reported")
                              & (F.col("_source_sha256").isNull() | (F.col("_source_sha256") == "") | F.col("_bronze_run_id").isNull()))
dq("DQ-COM-04", TBL_UNIT_YEAR, "BLOCK", no_lineage.count(), "lineage on every unit-year with values", f"{no_lineage.count()} missing",
   sample(no_lineage, "unit_psgc_code", "year"))

bad_cast = rows.filter(" OR ".join(f"({m}_text IS NOT NULL AND {m} IS NULL)" for m in MEASURES)
                       + " OR " + " OR ".join(f"{m}_text IS NULL" for m in MEASURES))
dq("DQ-COM-06", TBL_ROWS, "BLOCK", bad_cast.count(), "every value is a number or '-'", f"{bad_cast.count()} rows with other text",
   sample(bad_cast, "year", "area_label", "total_travelers_text"))

allowed = ["reported", "dash_as_zero", "reported_partial", "not_reported"]
null_any = " OR ".join(f"{m} IS NULL" for m in MEASURES)
bad_status = unit_year.filter(~F.col("traveler_value_status").isin(allowed)
                              | ((F.col("traveler_value_status") == "not_reported") & F.expr(f"NOT ({' AND '.join(f'{m} IS NULL' for m in MEASURES)})"))
                              | ((F.col("traveler_value_status") != "not_reported") & F.expr(null_any)))
dq("DQ-COM-07", TBL_UNIT_YEAR, "BLOCK", bad_status.count(), "NULL values exactly when not_reported", f"{bad_status.count()} violations",
   sample(bad_status, "unit_psgc_code", "year", "traveler_value_status"))

bad_sum = rows.filter(~F.col("row_total_is_dash")
                      & (F.col("domestic_travelers") + F.col("foreign_travelers") + F.col("overseas_filipinos") != F.col("total_travelers")))
dq("DQ-DOT-01", TBL_ROWS, "BLOCK", bad_sum.count(), "groups sum to total in every printed row", f"{bad_sum.count()} mismatches",
   sample(bad_sum, "year", "area_label", "total_travelers"))

no_role = rows.filter(F.col("row_role").isNull())
dq("DQ-DOT-03", TBL_ROWS, "BLOCK", no_role.count(), "every level-0/1 row has a crosswalk role", f"{no_role.count()} without a role",
   sample(no_role, "year", "parent_label", "area_label"),
   notes="Add new or renamed labels to resources/seeds/map_dot_area_to_psgc.csv and rerun 01_silver_geography.")

sub_rows = rows.filter(~F.col("indent_level").isin(0, 1))
dq("DQ-DOT-04", TBL_ROWS, "BLOCK", sub_rows.count(), "only indent levels 0 and 1", f"{sub_rows.count()} other rows")

def _row_total(year, label):
    r = rows.filter((F.col("year") == year) & (F.col("area_label") == label) & (F.col("indent_level") == 1)).select("total_travelers").collect()
    return r[0][0] if r else None
def _unit_total(year, unit):
    r = unit_year.filter((F.col("year") == year) & (F.col("unit_psgc_code") == unit)).select("total_travelers").collect()
    return r[0][0] if r else None
aklan_checks = []
if 2019 in dot_years:
    aklan_checks.append((2019, _unit_total(2019, "0600400000"), (_row_total(2019, "Aklan") or 0) + (_row_total(2019, "Boracay") or 0)))
if 2023 in dot_years:
    aklan_checks.append((2023, _unit_total(2023, "0600400000"), _row_total(2023, "Aklan")))
aklan_bad = [c for c in aklan_checks if c[1] != c[2]]
dq("DQ-DOT-07", TBL_UNIT_YEAR, "BLOCK", len(aklan_bad), "Aklan 2019 = Aklan + Boracay; 2023 = Aklan row",
   "; ".join(f"{y}: unit {u:,} vs expected {e:,}" if u is not None and e is not None else f"{y}: {u} vs {e}" for y, u, e in aklan_checks))

barmm_bad = unit_year.filter(F.col("unit_psgc_code").isin(list(EXPECTED_UNITS_WITHOUT_DOT))
                             & ((F.col("traveler_value_status") != "not_reported") | F.col("total_travelers").isNotNull()))
dq("DQ-DOT-09", TBL_UNIT_YEAR, "BLOCK", barmm_bad.count(), "BARMM provinces and Sulu are not_reported (NULL)",
   f"{barmm_bad.count()} violations", sample(barmm_bad, "unit_name", "year", "total_travelers"))

r2 = unit_year.filter("foreign_includes_overseas_filipinos").select("unit_name").distinct()
dq("DQ-DOT-10", TBL_UNIT_YEAR, "LOG", 0, "Region II units flagged", f"{r2.count()} units flagged: {sorted(r['unit_name'] for r in r2.collect())}")

w = Window.partitionBy("unit_psgc_code").orderBy("year")
yoy = (unit_year.filter(F.col("traveler_value_status") != "not_reported")
       .withColumn("prev_year", F.lag("year").over(w)).withColumn("prev_total", F.lag("total_travelers").over(w))
       .filter((F.col("prev_year") == F.col("year") - 1) & (F.col("prev_total") > 0) & (F.col("total_travelers") > 0)
               & ~F.col("year").isin(PANDEMIC_SWING_YEARS))
       .withColumn("ratio", F.col("total_travelers") / F.col("prev_total"))
       .filter((F.col("ratio") > YOY_RATIO_REVIEW) | (F.col("ratio") < 1 / YOY_RATIO_REVIEW))
       .withColumn("ratio_txt", F.format_number("ratio", 1)).orderBy(F.desc("ratio")))
dq("DQ-DOT-11", TBL_UNIT_YEAR, "REVIEW", yoy.count(), f"no year-over-year change above {YOY_RATIO_REVIEW:g}x (2022->2023 onward)",
   f"{yoy.count()} unit-years", sample(yoy, "unit_name", "prev_year", "year", "ratio_txt"),
   notes="Changes into 2020-2022 are not flagged (pandemic). Listed units need a source check (OI-5, e.g. Zamboanga Sibugay 2023-2024).")

recon = (rows.groupBy("year").agg(
            F.sum(F.when(F.col("row_role") == "national_total", F.col("total_travelers"))).alias("grand_total"),
            F.sum(F.when((F.col("row_role") == "exclude") & (F.col("exclude_reason") != "double_count"),
                         F.col("total_travelers"))).alias("excluded_not_double"))
         .join(unit_year.groupBy("year").agg(F.sum("total_travelers").alias("units_total")), "year")
         .withColumn("difference", F.col("units_total") + F.coalesce("excluded_not_double", F.lit(0)) - F.col("grand_total"))
         .orderBy("year"))
display(recon)
recon_bad = recon.filter((F.col("difference") != 0) | F.col("difference").isNull())
dq("DQ-DOT-15", TBL_UNIT_YEAR, "BLOCK", recon_bad.count(), "difference = 0 for every year",
   "; ".join(f"{r['year']}: {r['difference']}" for r in recon.collect()), sample(recon_bad, "year", "difference"),
   notes="Unit totals use '-' cells as 0 and leave out '-' rows (0 in DOT's own totals), so they reconcile exactly.")

counts = {r["traveler_value_status"]: r["count"] for r in unit_year.groupBy("traveler_value_status").count().collect()}
nr_from_dash = unit_year.filter((F.col("traveler_value_status") == "not_reported") & (F.col("contributing_rows") > 0)).count()
partial = counts.get("reported_partial", 0)
dq("DQ-DOT-16", TBL_UNIT_YEAR, "REVIEW", abs(nr_from_dash - 28) + abs(partial - 7),
   "28 not_reported from '-' rows, 7 reported_partial (DL-019, 2019-2024)",
   f"{nr_from_dash} not_reported from '-' rows, {partial} reported_partial; all statuses: {counts}",
   notes="Expected counts are for DOT 2019-2024; update them when a new DOT year is added.")

not_comp = {(r["unit_psgc_code"], r["year"]) for r in unit_year.filter(~F.col("comparable_to_2019")).select("unit_psgc_code", "year").collect()}
diff = not_comp ^ {k for k in EXPECTED_NOT_COMPARABLE if k[1] in dot_years}
dq("DQ-DOT-17", TBL_UNIT_YEAR, "REVIEW", len(diff), "comparable_to_2019 = false only for Aklan 2021-2024 (DL-020)",
   f"{len(not_comp)} unit-years not comparable", sorted(diff))

dq_df = spark.createDataFrame(dq_results,
    "run_id STRING, notebook_name STRING, check_id STRING, table_name STRING, severity STRING, status STRING, "
    "failed_rows BIGINT, expected STRING, observed STRING, sample_keys STRING, notes STRING"
    ).withColumn("checked_at", F.current_timestamp())
display(dq_df.select("check_id", "severity", "status", "failed_rows", "expected", "observed", "sample_keys"))

## Step 5 — Write Silver tables and update the run log

* All check results are appended to `dq_results`.
* If any `BLOCK` check failed: nothing is written, the run is logged as `FAILED`, and the notebook raises an error.
* Otherwise the three tables are overwritten.

In [0]:
dq_df.select("run_id", "notebook_name", "check_id", "table_name", "severity", "status", "failed_rows",
             "expected", "observed", "sample_keys", "checked_at", "notes") \
     .write.format("delta").mode("append").saveAsTable(dq_table("dq_results"))

block_failures = [r for r in dq_results if r[4] == "BLOCK" and r[5] == "FAIL"]
review_flags = [r for r in dq_results if r[4] == "REVIEW" and r[5] == "FAIL"]

def update_run_log(status, tables_written, rows_written, notes):
    spark.sql(f"""
    UPDATE {silver_table('silver_run_log')}
    SET status = :status, completed_at = current_timestamp(), tables_written = :tw, rows_written = :rw,
        dq_checks_run = :dc, dq_block_failures = :db, dq_review_flags = :dr, notes = :notes
    WHERE run_id = :run_id
    """, args={"status": status, "tw": tables_written, "rw": rows_written, "dc": len(dq_results),
               "db": len(block_failures), "dr": len(review_flags), "notes": notes, "run_id": run_id})

if block_failures:
    msg = "BLOCK checks failed: " + ", ".join(r[2] for r in block_failures) + ". No Silver tables written."
    update_run_log("FAILED", 0, 0, msg)
    raise AssertionError(msg + f" See {dq_table('dq_results')} for run_id = '{run_id}'.")

rows_out = rows.select("year", "page_number", "pdf_line_number", "indent_level", "parent_label", "area_label",
                       *MEASURES, *[f"{m}_is_dash" for m in MEASURES], "row_total_is_dash", "dash_cells",
                       "row_role", "unit_psgc_code", "belongs_to_unit", "area_psgc_code", "exclude_reason",
                       "breaks_comparability", "map_valid_from_year", "foreign_includes_overseas_filipinos",
                       "_source_dataset_id", "_source_file", "_source_sha256", "_bronze_run_id", "bronze_table",
                       "_silver_run_id", "_processed_at")
written = {
    TBL_ROWS: write_silver(rows_out, TBL_ROWS),
    TBL_UNIT_YEAR: write_silver(unit_year, TBL_UNIT_YEAR),
    TBL_EXCLUDED: write_silver(excluded, TBL_EXCLUDED),
}
for tbl, n in written.items():
    print(f"✓ {silver_table(tbl)}: {n} rows")

final_status = "SUCCESS_WITH_REVIEW" if review_flags else "SUCCESS"
final_notes = (f"Silver DOT: {', '.join(f'{k}={v}' for k, v in written.items())}; "
               f"{len(dq_results)} checks, {len(review_flags)} for review"
               + (f" ({', '.join(r[2] for r in review_flags)})" if review_flags else "") + ".")
update_run_log(final_status, len(written), sum(written.values()), final_notes)
print(f"Run {run_id}: {final_status}\n{final_notes}")
display(spark.table(silver_table("silver_run_log")).filter(F.col("run_id") == run_id))

## Preview

In [0]:
uy = spark.table(silver_table(TBL_UNIT_YEAR))
display(uy.filter(F.col("traveler_value_status") != "reported")
          .select("unit_name", "year", "total_travelers", "traveler_value_status", "comparable_to_2019", "dot_labels")
          .orderBy("traveler_value_status", "unit_name", "year"))
display(uy.filter(F.col("year").isin(2019, 2023)).groupBy("year")
          .agg(F.count("*").alias("units"), F.sum("total_travelers").alias("travelers")).orderBy("year"))